In [ ]:
import json
import itertools
import logging
import math
import os
import random
from pathlib import Path

import accelerate
import numpy as np
import safetensors  #
import torch
import torch.nn.functional as F
import transformers
from accelerate import Accelerator
from accelerate.logging import get_logger
from accelerate.utils import ProjectConfiguration, set_seed
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import AutoTokenizer, CLIPTextModel
from safetensors.torch import load_file

import diffusers
# from diffusers.pipelines import BlipDiffusionPipeline
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel, DiffusionPipeline
from diffusers.loaders import AttnProcsLayers
# from diffusers.models.attention_processor import CustomDiffusionAttnProcessor, CustomDiffusionAttnProcessor2_0
from diffusers.models.attention_processor import CustomDiffusionAttnProcessor
from diffusers.optimization import get_scheduler
from diffusers.utils import load_image

In [ ]:
class CustomDiffusionDataset(Dataset):
    """
    一个用于准备实例和类别图像以及提示词，以对模型进行微调的数据集。它对图像进行预处理，并对提示词进行分词。
    """

    def __init__(
        self,
        concepts_list,
        tokenizer,
        size=512,
        mask_size=64,
        center_crop=False,
        with_prior_preservation=False,
        num_class_images=200,
        hflip=False,
        aug=True,
    ):
        self.size = size
        self.mask_size = mask_size
        self.center_crop = center_crop
        self.tokenizer = tokenizer
        self.interpolation = Image.BILINEAR
        self.aug = aug
        # 实例图像路径
        self.instance_images_path = []
        # self.class_images_path = []
        self.with_prior_preservation = with_prior_preservation
        # 遍历图片和提示进行
        for concept in concepts_list:
            inst_img_path = [
                (x, concept["instance_prompt"]) for x in Path(concept["instance_data_dir"]).iterdir() if x.is_file()
            ]
            self.instance_images_path.extend(inst_img_path)
        # 打乱实例图像，避免训练顺序固定
        random.shuffle(self.instance_images_path)
        # 计算数量
        self.num_instance_images = len(self.instance_images_path)
        # self.num_class_images = len(self.class_images_path)
        # 计算长度；决定数据集长度
        # self._length = max(self.num_class_images, self.num_instance_images)
        self._length = self.num_instance_images
        # 水平翻转；50%概率
        self.flip = transforms.RandomHorizontalFlip(0.5 * hflip)
        # 数据增强
        self.image_transforms = transforms.Compose(
            [
                self.flip,
                # 缩放到size
                transforms.Resize(size, interpolation=transforms.InterpolationMode.BILINEAR),
                transforms.CenterCrop(size) if center_crop else transforms.RandomCrop(size),
                # 把HWC uint8 [0,255]转成CHW float32[0,1]
                transforms.ToTensor(),
                # 将数据变成【-1，1】
                transforms.Normalize([0.5], [0.5]),
            ]
        )

    def __len__(self):
        return self._length

    def preprocess(self, image, scale, resample):
        outer, inner = self.size, scale
        factor = self.size // self.mask_size
        if scale > self.size:
            outer, inner = scale, self.size
        top, left = np.random.randint(0, outer - inner + 1), np.random.randint(0, outer - inner + 1)
        image = image.resize((scale, scale), resample=resample)
        image = np.array(image).astype(np.uint8)
        image = (image / 127.5 - 1.0).astype(np.float32)
        instance_image = np.zeros((self.size, self.size, 3), dtype=np.float32)
        mask = np.zeros((self.size // factor, self.size // factor))
        if scale > self.size:
            instance_image = image[top : top + inner, left : left + inner, :]
            mask = np.ones((self.size // factor, self.size // factor))
        else:
            instance_image[top : top + inner, left : left + inner, :] = image
            mask[
                top // factor + 1 : (top + scale) // factor - 1, left // factor + 1 : (left + scale) // factor - 1
            ] = 1.0
        return instance_image, mask

    def __getitem__(self, index):
        example = {}
        instance_image, instance_prompt = self.instance_images_path[index % self.num_instance_images]
        instance_image = Image.open(instance_image)
        if not instance_image.mode == "RGB":
            instance_image = instance_image.convert("RGB")
        instance_image = self.flip(instance_image)

        # apply resize augmentation and create a valid image region mask
        random_scale = self.size
        if self.aug:
            random_scale = (
                np.random.randint(self.size // 3, self.size + 1)
                if np.random.uniform() < 0.66
                else np.random.randint(int(1.2 * self.size), int(1.4 * self.size))
            )
        instance_image, mask = self.preprocess(instance_image, random_scale, self.interpolation)

        if random_scale < 0.6 * self.size:
            instance_prompt = np.random.choice(["a far away ", "very small "]) + instance_prompt
        elif random_scale > self.size:
            instance_prompt = np.random.choice(["zoomed in ", "close up "]) + instance_prompt

        example["instance_images"] = torch.from_numpy(instance_image).permute(2, 0, 1)
        example["mask"] = torch.from_numpy(mask)
        example["instance_prompt_ids"] = self.tokenizer(
            instance_prompt,
            truncation=True,
            padding="max_length",
            max_length=self.tokenizer.model_max_length,
            return_tensors="pt",
        ).input_ids

        return example


In [ ]:
def collate_fn(examples):
    """
    整合一批数据。
    参数：
        examples（字典列表）：一个字典列表，其中每个字典包含键“instance_prompt_ids”“instance_images”和“mask”。实例提示标识；实例图像；掩码
    返回：
        dict：一个包含批量“input_ids”“pixel_values”和“mask”的字典。
              “input_ids”是提示词标记ID的拼接张量。
              “pixel_values”是实例图像的堆叠张量。
              “mask”是添加了通道维度的掩码堆叠张量。
    """
    input_ids = [example["instance_prompt_ids"] for example in examples]
    pixel_values = [example["instance_images"] for example in examples]
    mask = [example["mask"] for example in examples]
    input_ids = torch.cat(input_ids, dim=0) # 沿着批次维度拼接提示词令牌ID。
    pixel_values = torch.stack(pixel_values) # 将单个图像张量堆叠起来以形成图像批次。
    mask = torch.stack(mask) # 将各个单独的掩码张量堆叠起来，形成掩码批次。
    pixel_values = pixel_values.to(memory_format=torch.contiguous_format).float() # 确保数据位于连续的内存中以实现可能更快的处理，并将其转换为浮点数。
    mask = mask.to(memory_format=torch.contiguous_format).float() # 确保掩码位于连续内存中并转换为浮点数。
    # 创建一个批次字典
    batch = {"input_ids": input_ids, "pixel_values": pixel_values, "mask": mask.unsqueeze(1)} # 为掩码添加一个通道维度，以与图像处理层兼容。
    return batch

In [ ]:
def save_new_embed(text_encoder, modifier_token_id, accelerator, modifier_token, output_dir, safe_serialization=True):
    """
    从训练好的文本编码器中提取新学习的令牌嵌入；并保存到文件中（用于推理或后续加载）
    保存文本编码器学习到的新的令牌嵌入。
    参数：
        text_encoder：经过训练的文本编码器模型。
        modifier_token_id（int列表）：与修饰符令牌对应的令牌ID列表。
        accelerator：用于分布式训练的加速器对象。
        modifier_token（str列表）：修饰符令牌的列表（例如，["<new1>"]）。
        output_dir（str）：将保存新嵌入的目录。
        safe_serialization（bool，可选）：是否使用安全序列化（safetensors）。默认为True。
    """
    # Unwrap the potentially distributed text encoder model to access its components.
    # 解开可能为分布式的文本编码器模型，以访问其组件。获取模型权重
    learned_embeds = accelerator.unwrap_model(text_encoder).get_input_embeddings().weight

    # Iterate through the modifier tokens and their corresponding IDs.
    # 遍历修饰符标记及其对应的ID。
    # for x, y in zip(modifier_token_id, modifier_token):
    for x, y in zip(modifier_token_id, [modifier_token]):
        # x：token ID；y：token字符串
        # Create a dictionary to store the learned embedding for the current modifier token.
        # 创建一个字典来存储当前修饰符标记的已学习嵌入。
        # 创建一个字典，把token字符串映射到对应的嵌入张量中
        learned_embeds_dict = {}
        # Extract the learned embedding for the specific modifier token ID.
        # 提取特定修饰符标记ID的习得嵌入。
        learned_embeds_dict[y] = learned_embeds[x]

        # Determine the filename and saving method based on the safe_serialization flag.
        # 根据safe_serialization标志确定文件名和保存方法。
        if safe_serialization:
            filename = os.path.join(output_dir, "learned_embeds.safetensors")
            safetensors.torch.save_file(
                learned_embeds_dict,
                filename,
                metadata={"format": "pt"}
            )
        else:
            filename = f"{output_dir}/{y}.bin"
            # Save the learned embedding dictionary using the standard PyTorch format.
            torch.save(learned_embeds_dict, filename)


In [ ]:
def train_func(
    output_dir,
    instance_prompt,
    instance_data_dir,
    freeze_model,
    learning_rate,
    max_train_steps,
    train_batch_size
):
    accelerator_project_config = ProjectConfiguration(project_dir=output_dir, logging_dir=Path(output_dir, "logs"))

    # 加速器
    accelerator = Accelerator(
        gradient_accumulation_steps=1,
        mixed_precision=None,
        log_with="tensorboard",
        project_config=accelerator_project_config,
    )


    # 我们需要初始化所使用的跟踪器，同时也要存储我们的配置。
    # 跟踪器会在主进程上自动初始化。
    if accelerator.is_main_process:
        accelerator.init_trackers("custom-diffusion")

    # 处理仓库创建
    if accelerator.is_main_process and (output_dir is not None):
        os.makedirs(output_dir, exist_ok=True)

    pretrained_model_name_or_path = "./stable-diffusion-v1-5"

    # 分词器
    tokenizer = AutoTokenizer.from_pretrained(
        pretrained_model_name_or_path,
        subfolder="tokenizer",
        revision=None,
        use_fast=False,
    )

    # 加载调度器和模型
    noise_scheduler = DDPMScheduler.from_pretrained(pretrained_model_name_or_path, subfolder="scheduler")
    text_encoder = CLIPTextModel.from_pretrained(
        pretrained_model_name_or_path, subfolder="text_encoder", revision=None, variant=None
    )
    vae = AutoencoderKL.from_pretrained(
        pretrained_model_name_or_path, subfolder="vae", revision=None, variant=None
    )
    unet = UNet2DConditionModel.from_pretrained(
        pretrained_model_name_or_path, subfolder="unet", revision=None, variant=None
    )

    # 添加一个经过优化的修饰符令牌
    modifier_token_id = []
    initializer_token_id = []
    modifier_token = "<new1>"
    initializer_token = "object"

    # 在分词器中添加占位符标记
    num_added_tokens = tokenizer.add_tokens(modifier_token)
    if num_added_tokens == 0:
        raise ValueError(
            f"The tokenizer already contains the token {modifier_token}. Please pass a different"
            " `modifier_token` that is not already in the tokenizer."
        )

    # 将initializer_token、placeholder_token转换为id
    token_ids = tokenizer.encode(initializer_token, add_special_tokens=False)

    # 检查initializer_token是单个标记还是标记序列
    if len(token_ids) > 1:
        raise ValueError("The initializer token must be a single token.")

    initializer_token_id.append(token_ids[0])
    modifier_token_id.append(tokenizer.convert_tokens_to_ids(modifier_token))

    # 由于我们要向分词器中添加新的特殊标记，所以需要调整标记嵌入的大小
    text_encoder.resize_token_embeddings(len(tokenizer))

    # 使用初始化器标记的嵌入来初始化新添加的占位符标记
    token_embeds = text_encoder.get_input_embeddings().weight.data
    for x, y in zip(modifier_token_id, initializer_token_id):
        token_embeds[x] = token_embeds[y]

    # 冻结除文本编码器中的令牌嵌入之外的所有参数
    params_to_freeze = itertools.chain(
        text_encoder.text_model.encoder.parameters(),
        text_encoder.text_model.final_layer_norm.parameters(),
        text_encoder.text_model.embeddings.position_embedding.parameters(),
    )
    for param in params_to_freeze:
        param.requires_grad = False

    vae.requires_grad_(False)
    if modifier_token is None:
        text_encoder.requires_grad_(False)
    unet.requires_grad_(False)

    # 对于混合精度训练，我们将文本编码器和变分自编码器的权重转换为半精度
    # 由于这些模型仅用于推理，保持权重为全精度并非必要。
    weight_dtype = torch.float32
    if accelerator.mixed_precision == "fp16":
        weight_dtype = torch.float16
    elif accelerator.mixed_precision == "bf16":
        weight_dtype = torch.bfloat16

    # 将unet、vae和text_encoder移至设备并转换为weight_dtype类型
    if accelerator.mixed_precision != "fp16" and modifier_token is not None:
        text_encoder.to(accelerator.device, dtype=weight_dtype)
    unet.to(accelerator.device, dtype=weight_dtype)
    vae.to(accelerator.device, dtype=weight_dtype)

    # attention_class = (
    #     CustomDiffusionAttnProcessor2_0 if hasattr(F, "scaled_dot_product_attention") else CustomDiffusionAttnProcessor
    # )
    attention_class = CustomDiffusionAttnProcessor
    
    train_kv = True
    train_q_out = False if freeze_model == "crossattn_kv" else True
    custom_diffusion_attn_procs = {}

    st = unet.state_dict()
    # 遍历unet所有注意力
    for name, _ in unet.attn_processors.items():
        cross_attention_dim = None if name.endswith("attn1.processor") else unet.config.cross_attention_dim
        if name.startswith("mid_block"):
            # 中间采样
            hidden_size = unet.config.block_out_channels[-1]
        elif name.startswith("up_blocks"):
            # 上采样
            block_id = int(name[len("up_blocks.")])
            hidden_size = list(reversed(unet.config.block_out_channels))[block_id]
        elif name.startswith("down_blocks"):
            # 下采样
            block_id = int(name[len("down_blocks.")])
            hidden_size = unet.config.block_out_channels[block_id]
        layer_name = name.split(".processor")[0]
        # 提取原始的权重
        weights = {
            "to_k_custom_diffusion.weight": st[layer_name + ".to_k.weight"],
            "to_v_custom_diffusion.weight": st[layer_name + ".to_v.weight"],
        }
        
        if train_q_out:
            weights["to_q_custom_diffusion.weight"] = st[layer_name + ".to_q.weight"]
            weights["to_out_custom_diffusion.0.weight"] = st[layer_name + ".to_out.0.weight"]
            weights["to_out_custom_diffusion.0.bias"] = st[layer_name + ".to_out.0.bias"]
        # 创建自定义扩散注意力
        if cross_attention_dim is not None:
            custom_diffusion_attn_procs[name] = attention_class(
                train_kv=train_kv,
                train_q_out=train_q_out,
                hidden_size=hidden_size,
                cross_attention_dim=cross_attention_dim,
            ).to(unet.device)
            custom_diffusion_attn_procs[name].load_state_dict(weights)
        else:
            custom_diffusion_attn_procs[name] = attention_class(
                train_kv=False,
                train_q_out=False,
                hidden_size=hidden_size,
                cross_attention_dim=cross_attention_dim,
            )
    del st
    # 替换初始化注意力
    unet.set_attn_processor(custom_diffusion_attn_procs)
    # 提取要训练的注意力
    custom_diffusion_layers = AttnProcsLayers(unet.attn_processors)
    accelerator.register_for_checkpointing(custom_diffusion_layers)

    # 调整学习率
    learning_rate = learning_rate * train_batch_size * accelerator.num_processes

    # 优化器
    optimizer = torch.optim.AdamW(
        itertools.chain(text_encoder.get_input_embeddings().parameters(), custom_diffusion_layers.parameters())
        if modifier_token is not None
        else custom_diffusion_layers.parameters(),
        lr=learning_rate,
        betas=(0.9, 0.999),
        weight_decay=1e-2,
        eps=1e-8
    )

    # Dataset and DataLoaders creation:.
    # 数据集和数据加载器的创建：/实例提示词/类别提示词/实例目录/类别目录
    concepts_list = [
        {
            "instance_prompt": instance_prompt,
            "class_prompt": None,
            "instance_data_dir": instance_data_dir,
            "class_data_dir": None,
        }
    ]
    resolution = 512
    # 创建数据集
    train_dataset = CustomDiffusionDataset(
        concepts_list=concepts_list,
        tokenizer=tokenizer,
        size=resolution,
        mask_size=vae.encode(
            torch.randn(1, 3, resolution, resolution).to(dtype=weight_dtype).to(accelerator.device)
        ).latent_dist.sample().size()[-1],
        center_crop=True,
        hflip=True,
        aug=True,
    )
    # 创建批次
    train_dataloader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=train_batch_size,
        shuffle=True,
        collate_fn=lambda examples: collate_fn(examples),
        num_workers=0,  
    )
    # 工作进程数量    
    # 关于训练步数的调度器和相关计算。
    num_warmup_steps_for_scheduler = 0
    num_training_steps_for_scheduler = max_train_steps * accelerator.num_processes
    
    lr_scheduler = get_scheduler(
        "constant",
        optimizer=optimizer,
        num_warmup_steps=num_warmup_steps_for_scheduler,
        num_training_steps=num_training_steps_for_scheduler,
    )

    # 用我们的`accelerator`准备好所有东西。
    if modifier_token is not None:
        custom_diffusion_layers, text_encoder, optimizer, train_dataloader, lr_scheduler = accelerator.prepare(
            custom_diffusion_layers, text_encoder, optimizer, train_dataloader, lr_scheduler
        )
    else:
        custom_diffusion_layers, optimizer, train_dataloader, lr_scheduler = accelerator.prepare(
            custom_diffusion_layers, optimizer, train_dataloader, lr_scheduler
        )

    # 由于训练数据加载器的大小可能发生了变化，我们需要重新计算总的训练步数。
    num_update_steps_per_epoch = len(train_dataloader)
    # 之后我们重新计算训练轮次的数量
    num_train_epochs = math.ceil(max_train_steps / num_update_steps_per_epoch)

    # Train!
    global_step = 0
    first_epoch = 0
    initial_global_step = 0

    # 进度条
    progress_bar = tqdm(
        range(0, max_train_steps),
        initial=initial_global_step,
        desc="Steps",
        disable=not accelerator.is_local_main_process,
    )

    for epoch in range(first_epoch, num_train_epochs):
        unet.train()
        if modifier_token is not None:
            text_encoder.train()
        for step, batch in enumerate(train_dataloader):
            with accelerator.accumulate(unet), accelerator.accumulate(text_encoder):
                # 将图像转换到潜在空间
                latents = vae.encode(batch["pixel_values"].to(dtype=weight_dtype)).latent_dist.sample()
                latents = latents * vae.config.scaling_factor

                # 我们将添加到潜在变量中的样本噪声
                noise = torch.randn_like(latents)
                bsz = latents.shape[0]
                # 为每张图像随机采样一个时间步长
                timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=latents.device)
                timesteps = timesteps.long()
                # 根据每个时间步的噪声幅度向潜在变量添加噪声
                #（这是前向扩散过程）
                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

                # 获取用于条件调节的文本嵌入
                encoder_hidden_states = text_encoder(batch["input_ids"])[0]

                # 预测噪声残差
                model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample

                # 根据预测类型获取损失目标
                if noise_scheduler.config.prediction_type == "epsilon":
                    target = noise
                elif noise_scheduler.config.prediction_type == "v_prediction":
                    target = noise_scheduler.get_velocity(latents, noise, timesteps)
                else:
                    raise ValueError(f"Unknown prediction type {noise_scheduler.config.prediction_type}")
                # 计算loss
                # mask = batch["mask"]
                # loss = F.mse_loss(model_pred.float(), target.float(), reduction="none")
                # loss = ((loss * mask).sum / mask.sum).mean()
                # loss = ((loss * mask).sum([1, 2, 3]) / mask.sum([1, 2, 3])).mean()
                loss = F.mse_loss(model_pred.float(), target.float())
                accelerator.backward(loss)
                # 除了为该概念新添加的嵌入之外，将所有标记嵌入的梯度清零
                # 因为我们只想优化概念嵌入
                if modifier_token is not None:
                    if accelerator.num_processes > 1:
                        grads_text_encoder = text_encoder.module.get_input_embeddings().weight.grad
                    else:
                        grads_text_encoder = text_encoder.get_input_embeddings().weight.grad
                    # 获取我们想要将梯度清零的标记的索引
                    # 只训练新的令牌嵌入
                    index_grads_to_zero = torch.arange(len(tokenizer)) != modifier_token_id[0]
                    for i in range(1, len(modifier_token_id)):
                        index_grads_to_zero = index_grads_to_zero & (
                            torch.arange(len(tokenizer)) != modifier_token_id[i]
                        )
                    grads_text_encoder.data[index_grads_to_zero, :] = grads_text_encoder.data[
                        index_grads_to_zero, :
                    ].fill_(0)

                if accelerator.sync_gradients:
                    params_to_clip = (
                        itertools.chain(text_encoder.parameters(), custom_diffusion_layers.parameters())
                        if modifier_token is not None
                        else custom_diffusion_layers.parameters()
                    )
                    accelerator.clip_grad_norm_(params_to_clip, 1.0)
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad(set_to_none=False)

            # 检查加速器是否在后台执行了优化步骤
            if accelerator.sync_gradients:
                progress_bar.update(1)
                global_step += 1

                if accelerator.is_main_process and (global_step == max_train_steps):
                    save_path = os.path.join(output_dir, f"checkpoint-{global_step}")
                    accelerator.save_state(save_path)

            logs = {"loss": loss.detach().item(), "lr": lr_scheduler.get_last_lr()[0]}
            progress_bar.set_postfix(**logs)
            accelerator.log(logs, step=global_step)

            if global_step >= max_train_steps:
                break

    # 保存自定义扩散层
    accelerator.wait_for_everyone()
    if accelerator.is_main_process:
        unet = unet.to(torch.float32)
        # 保存内容
        unet.save_attn_procs(output_dir, safe_serialization=False)
        save_new_embed(
            text_encoder,
            modifier_token_id,
            accelerator,
            modifier_token,
            output_dir,
            safe_serialization=True,
        )

    accelerator.end_training()

In [ ]:
instance_prompt = "photo of a <new1> Spring Festival image with zodiac signs as the main elements, in a two-dimensional or anime style"    
instance_data_dir = "data/object-2"     # 要自定义的对象的图像路径
parameter_to_train = "crossattn_kv"  ]  # “crossattn_kv”仅训练交叉注意力中的K和V。如果你还想训练Q，请将其改为“crossattn”
learning_rate = 2e-5
max_train_steps = 1200
train_batch_size = 1

ckpt_dir = "output"                               # 用于保存检查点的目录名称


In [ ]:
train_func(
    ckpt_dir,
    instance_prompt,
    instance_data_dir,
    parameter_to_train,
    learning_rate,
    max_train_steps,
    train_batch_size
)

In [ ]:
pipe = DiffusionPipeline.from_pretrained(
    "./stable-diffusion-v1-5", torch_dtype=torch.float16
).to("cuda")

pipe.unet.load_attn_procs(
    ckpt_dir,
    weight_name="pytorch_custom_diffusion_weights.bin"
)

In [ ]:
generate_prompt = "A <new1>New Year image</new1> with a dog as the main element, in a 2D or anime style, and a festive background"  

num_inference_steps = 200   # 去噪步骤的数量。更多的去噪步骤通常会带来更高质量的图像，但会以更慢的推理速度为代价。
guidance_scale = 10.0        # 较高的引导尺度有助于生成与文本提示紧密相关的图像，但这通常是以降低图像质量为代价的。

obj = instance_data_dir.split("/")[-1]
output_dir = "results"
os.makedirs(f"{output_dir}/{obj}", exist_ok = True)

for i in range(2):
    image = pipe(
        generate_prompt,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        eta=1.0,
    ).images[0]
    image.save(f"{output_dir}/{obj}/{i}.jpg")